# FAR-Trans HW2 — LLM attribute encoder (Anastasia Getman, Group 7)

Runs the four provided baselines, my LLM attribute-extraction encoder (with its raw-text and
rule-based same-information controls) and, optionally, the course paraphrase encoder on the
same 100 users, cutoff 2021-06-01, 30-day window.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Put `FAR-Trans.zip` in Google
Drive at `MyDrive/far-trans-hw2/FAR-Trans.zip` (or upload it when cell 3 asks).
Outputs and the generation cache are written to Drive, so a disconnect does not lose progress:
re-run all cells and the encoder continues from its cache (then the timing is no longer a cold run).

In [ ]:
# 1. GPU and Drive
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
WORK = pathlib.Path('/content/drive/MyDrive/far-trans-hw2'); WORK.mkdir(parents=True, exist_ok=True)
OUT = WORK / 'outputs'; OUT.mkdir(exist_ok=True)
print(WORK)

In [ ]:
# 2. Code: course repo pinned to the starter commit, plus my team repository
COURSE_COMMIT = 'f6850861ea20a51e1fb018ee4e5f840c2ac759e0'
TEAM_REPO = ''   # e.g. 'https://github.com/getmanyan/far-trans-group7.git' (leave empty to upload files)
!rm -rf /content/RecSys-LLMs && git clone -q https://github.com/dryjins/RecSys-LLMs.git /content/RecSys-LLMs
!git -C /content/RecSys-LLMs checkout -q {COURSE_COMMIT} && git -C /content/RecSys-LLMs log -1 --format='course commit %H %cd'
COURSE = '/content/RecSys-LLMs/project'
if TEAM_REPO:
    !rm -rf /content/team && git clone -q {TEAM_REPO} /content/team
    CODE = '/content/team/getman/hw2'
    !git -C /content/team log -1 --format='my commit %H %cd'
else:
    from google.colab import files
    CODE = '/content/hw2'; os.makedirs(CODE, exist_ok=True); os.chdir(CODE)
    print('Upload attribute_encoder.py, run_course_encoder.py, analyze.py, test_attribute_encoder.py')
    files.upload()
os.chdir(CODE); print(sorted(os.listdir(CODE)))

In [ ]:
# 3. Data
import zipfile
ZIP = WORK / 'FAR-Trans.zip'
if not ZIP.exists():
    from google.colab import files
    up = files.upload()            # choose FAR-Trans.zip
    ZIP.write_bytes(up[next(iter(up))])
with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content/data')
DATA = '/content/data/FAR-Trans'
!ls -la {DATA}
!sha256sum {DATA}/transactions.csv {DATA}/asset_information.csv

In [ ]:
# 4. Dependencies (Colab's preinstalled torch is kept; versions are recorded in run_manifest.json)
!pip -q install transformers==4.57.6 huggingface_hub
import torch, transformers, platform
print('python', platform.python_version(), '| torch', torch.__version__, '| transformers', transformers.__version__,
      '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
      '| capability', torch.cuda.get_device_capability() if torch.cuda.is_available() else None)

In [ ]:
# 5. Tests (no model, no real data)
!cd {COURSE}/far-trans && python -m unittest discover -s . -p 'test_*.py' 2>&1 | tail -2
!cd {COURSE}/far-trans-llm && python -m unittest discover -s . -p 'test_*.py' 2>&1 | tail -2
!cd {CODE} && python -m unittest test_attribute_encoder 2>&1 | tail -2

In [ ]:
# 6. Four provided baselines (CPU, a few seconds)
import time, subprocess
t = time.perf_counter()
!cd {COURSE}/far-trans && python starter.py run --data-dir {DATA} --output-dir {OUT}/baseline
BASELINE_SECONDS = time.perf_counter() - t
(OUT / 'baseline_seconds.txt').write_text(f'{BASELINE_SECONDS:.2f}\n')
print(f'baseline run: {BASELINE_SECONDS:.1f} s')

In [ ]:
# 7. Download the pinned model once (download time is NOT part of the measured run)
from huggingface_hub import snapshot_download
MODEL_ID, REVISION = 'Qwen/Qwen3-1.7B', '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'
MODEL = snapshot_download(MODEL_ID, revision=REVISION)
print(MODEL)

In [ ]:
# 8. MY METHOD: LLM attribute encoder (+ raw-text and rule controls, attributes-only ablation)
# ~767 short generations; expect roughly 15-40 min on a T4. Safe to re-run after a disconnect.
!cd {CODE} && python attribute_encoder.py --course-dir {COURSE} --data-dir {DATA} \
    --output-dir {OUT}/attr_encoder --model {MODEL} --revision {REVISION} \
    --device cuda --dtype auto --on-invalid fallback 2>&1 | tail -15

In [ ]:
# 9. Quick look: invalid outputs, coverage, agreement with rule-based attributes, examples
import json
m = json.load(open(OUT / 'attr_encoder' / 'run_manifest.json'))['runtime']
for k in ['model', 'resolved_revision', 'gpu', 'dtype', 'torch_version', 'transformers_version',
          'new_calls', 'cache_hits', 'invalid_outputs', 'assets_without_known_attribute',
          'inference_seconds', 'end_to_end_seconds', 'known_values', 'llm_rule_agreement']:
    print(f'{k}: {m.get(k)}')
for line in open(OUT / 'attr_encoder' / 'asset_attributes.jsonl').readlines()[:8]:
    print(json.loads(line))

## Optional: course paraphrase encoder on the same GPU

Lets you compare *paraphrase* vs *attribute extraction* under identical hardware and dtype
(the course's own numbers were measured in bfloat16 on an RTX 5070 Ti with cache reuse).
Another ~767 generations (up to 160 tokens each); expect roughly 20-60 min on a T4.

In [ ]:
# 10. (optional) Course LLM-Rec-inspired paraphrase encoder, unchanged prompt and settings
RUN_PARAPHRASE = True
if RUN_PARAPHRASE:
    !cd {CODE} && python run_course_encoder.py --course-dir {COURSE} --dtype auto --data-dir {DATA} \
        --output-dir {OUT}/paraphrase_encoder --model {MODEL} --revision {REVISION} \
        --device cuda --on-invalid fallback 2>&1 | tail -10

In [ ]:
# 11. Results table (4 decimals), subgroup split, protocol checks
runs = [str(OUT / 'attr_encoder')] + ([str(OUT / 'paraphrase_encoder')]
        if (OUT / 'paraphrase_encoder' / 'metrics.json').exists() else [])
RUNS = ' '.join(runs)
bs = (OUT / 'baseline_seconds.txt').read_text().strip()
!cd {CODE} && python analyze.py --course-dir {COURSE} --data-dir {DATA} --baseline {OUT}/baseline \
    --runs {RUNS} --baseline-seconds {bs} --out {OUT}/results.md

In [ ]:
# 12. Pack everything for the report (download this zip and send it to Claude)
!cd {WORK} && rm -f hw2_outputs.zip && zip -qr hw2_outputs.zip outputs && ls -la hw2_outputs.zip
from google.colab import files
files.download(str(WORK / 'hw2_outputs.zip'))